# 🎬 AI Movie Maker Colab（MiniMax H3-Turbo × autoip-mv-maker 整合版）

這份 Notebook 將 **[autoip/autoip-mv-maker](https://github.com/autoip/autoip-mv-maker)** 的分鏡腳本架構與 **`MiniMax_H3_Turbo_Colab.ipynb` (`1zdxzAm834HesHG2uu7GbM4Kv0kbKOLLu`)** 的真實 ComfyUI 節點工作流完整整合，針對 Colab A100 部署與四大痛點設計：

1. **🛡️ 畫面連貫不跑掉（Visual Continuity & Anti-Drift）**
   - 支援官方 `MiniMaxH3ReferenceToVideo`（Ref2VA，最多 9 張 `<Picture 1>`–`<Picture 9>`）與 `MiniMaxH3ImageToVideo`（FL2VA 首幀模式），畫布對齊官方 `1344×768`（32 倍數）與 `17k+5` 影格數規則（24 fps）。
   - 支援 **`continuity_mode = "hybrid" | "chain_last_frame" | "anchor"`**：同一場戲自動擷取上一鏡在 `tail_guard`（裁除最後 `0.35s` 易變形閃白尾幀）之前的乾淨穩定幀作為下一鏡的 `<Picture 1>`，同時保留 `<Picture 2..N>` 角色設定圖。
2. **🔊 嚴格單一聲軌（Single Audio Track Policy）**
   - 提供三種互斥音軌模式，從 Prompt 階段與 FFmpeg 封裝階段雙重把關，並以 `ffprobe` 強制驗證成品只有 **1 條視訊軌 + 1 條音訊軌**：
     - `native_h3_audio`：只使用 MiniMax H3 原生單一語音（`<d>[Chinese] ...</d>`），每鏡自動做 `-16 LUFS` 響度正規化（`loudnorm`）。
     - `clip_voiceover`：Prompt 禁止模型發出人聲，後製以外部旁白 WAV 完全替換（`-map 0:v:0 -map 1:a:0`），絕不疊加雙重人聲。
     - `master_audio`：所有生成片段以 `-an` 移除音軌，最後只鋪上唯一一條主音軌（Master Audio）。
3. **📋 分鏡腳本直接生成（Storyboard-Driven）**
   - 可直接貼上 **Markdown 分鏡表格**、上傳 **`.md` / `.json`**，或讀取 `autoip-mv-maker` 的 **`project.py`**。每鏡長度自動檢查（單鏡上限 `14.65s`，加 `0.35s` tail guard 不超過 H3 官方 15 秒上限）。
4. **📝 自動時間軸對齊與燒錄繁體中文字幕（Auto Subtitles）**
   - 自動依標點切分繁體中文字句，支援釋放 ComfyUI 顯存後以 `faster-whisper` 對齊時間軸，輸出 `.zh-TW.srt`、`.ass`、無字幕版 `_clean.mp4`、燒錄字幕版 `_subtitled.mp4` 與全片連貫性檢視圖 `_review_sheet.jpg`。

In [ ]:
#@title Step 1. 檢查 Colab GPU 環境、掛載 Google Drive、安裝 ComfyUI + VideoHelperSuite + AI-Movie
MOUNT_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/ComfyUI_MiniMax_H3" #@param {type:"string"}
INSTALL_FASTER_WHISPER = False #@param {type:"boolean"}
REPO_URL = "https://github.com/a017749-chen/AI-movie.git" #@param {type:"string"}

import os
import shutil
import subprocess
import sys
from pathlib import Path
import torch

assert torch.cuda.is_available(), "請在 Colab「執行階段 > 變更執行階段類型」選擇 GPU（建議 A100）。"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("PyTorch CUDA:", torch.version.cuda)
print("Free disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        Path(DRIVE_ROOT, "models").mkdir(parents=True, exist_ok=True)
        Path(DRIVE_ROOT, "outputs").mkdir(parents=True, exist_ok=True)
        print(f"[OK] Google Drive 已掛載：{DRIVE_ROOT}")
    except Exception as exc:
        print(f"[WARN] 未掛載 Google Drive（{exc}），將使用 /content 本機磁碟。")
        MOUNT_GOOGLE_DRIVE = False

# 1. 安裝繁體中文字型 (Noto Sans CJK TC 供 FFmpeg libass 燒錄字幕) 與 ffmpeg
subprocess.run(["apt-get", "-qq", "update"], check=False)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "fonts-noto-cjk"], check=False)
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "需要 ffmpeg 與 ffprobe。"

# 2. 安裝官方 Comfy-Org/ComfyUI 與 Kosinkadink/ComfyUI-VideoHelperSuite
COMFY = Path("/content/ComfyUI")
VHS = COMFY / "custom_nodes" / "ComfyUI-VideoHelperSuite"
SETUP_MARKER = Path("/content/.h3_comfyui_setup_v1")
if not SETUP_MARKER.exists():
    if not COMFY.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Comfy-Org/ComfyUI.git", str(COMFY)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt"),
         "huggingface_hub", "hf_xet", "pillow", "requests"],
        check=True,
    )
    if not VHS.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git", str(VHS)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(VHS / "requirements.txt")], check=True)
    SETUP_MARKER.touch()
else:
    print("重用本次 Colab session 已安裝的 ComfyUI 環境")

# 3. Clone 或更新 AI-movie (mvkit)
REPO_DIR = Path("/content/AI-movie")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if INSTALL_FASTER_WHISPER:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=False)

WORKSPACE_DIR = Path("/content/workspace_aimovie")
REFS_DIR = WORKSPACE_DIR / "refs"
AUDIO_DIR = WORKSPACE_DIR / "audio"
OUTPUT_DIR = WORKSPACE_DIR / "outputs"
for d in (WORKSPACE_DIR, REFS_DIR, AUDIO_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("[READY] 環境安裝完成！")

In [ ]:
#@title Step 2. 上傳參考圖（`<Picture 1>` 場景基準圖、`<Picture 2..N>` 角色設定圖）與外部音檔（選填）
UPLOAD_REFERENCE_IMAGES = False #@param {type:"boolean"}
UPLOAD_AUDIO_FILES = False #@param {type:"boolean"}

from pathlib import Path
from PIL import Image, ImageDraw

if UPLOAD_REFERENCE_IMAGES:
    from google.colab import files
    print("請上傳 1–9 張參考圖片（例如 scene_anchor.png, char_learner.png, char_mentor.png）：")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = REFS_DIR / Path(fname).name
        dst.write_bytes(data)
        print(f"已儲存參考圖：{dst}")

if UPLOAD_AUDIO_FILES:
    from google.colab import files
    print("請上傳單一主音軌（master_audio 模式）或各鏡旁白 WAV（clip_voiceover 模式）：")
    uploaded = files.upload()
    for fname, data in uploaded.items():
        dst = AUDIO_DIR / Path(fname).name
        dst.write_bytes(data)
        print(f"已儲存音檔：{dst}")

existing_imgs = sorted(
    p for p in REFS_DIR.iterdir()
    if p.suffix.lower() in (".png", ".jpg", ".jpeg", ".webp")
)
default_anchor = REFS_DIR / "scene_anchor.png"
if not existing_imgs:
    img = Image.new("RGB", (1344, 768), color=(34, 45, 62))
    draw = ImageDraw.Draw(img)
    draw.rectangle([80, 64, 1264, 704], outline=(180, 200, 220), width=4)
    img.save(default_anchor)
    existing_imgs = [default_anchor]
    print(f"[INFO] 尚未上傳圖片，已建立預設 1344×768 場景基準圖：{default_anchor}")
else:
    default_anchor = existing_imgs[0]
    print(f"[OK] 目前可用參考圖：{[p.name for p in existing_imgs]}")

In [ ]:
#@title Step 3. 載入分鏡腳本、設定「單一聲軌」與「防跑掉連貫模式」並預覽 H3 Prompt
PROJECT_TITLE = "AI_Movie_Output" #@param {type:"string"}
STORYBOARD_SOURCE_TYPE = "markdown_inline" #@param ["markdown_inline", "file_path", "upload_file"]
STORYBOARD_FILE_PATH = "/content/AI-movie/examples/storyboard_template.md" #@param {type:"string"}

#@markdown ---
#@markdown ### 🔊 單一聲軌設定（保證最終影片只有唯一一條音軌，絕不雙聲軌疊加）
AUDIO_MODE = "native_h3_audio" #@param ["native_h3_audio", "clip_voiceover", "master_audio"]
MASTER_AUDIO_PATH = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 🛡️ 畫面防跑掉與連貫性設定（Visual Continuity & Anti-Drift）
INFERENCE_MODE = "reference" #@param ["reference", "first_frame"]
CONTINUITY_MODE = "hybrid" #@param ["hybrid", "chain_last_frame", "anchor"]
REF_IMAGE_SIZE = "max" #@param ["max", "match"]
ASPECT_RATIO = "16:9" #@param ["16:9", "9:16", "1:1"]
TAIL_GUARD_SEC = 0.35 #@param {type:"number"}
BASE_SEED = 768914106098754 #@param {type:"integer"}

# 可直接在此貼上你的 Markdown 分鏡表格（單鏡建議 5–14 秒）：
MARKDOWN_STORYBOARD = '''
| 鏡號 | 秒數 | 場景 | 畫面與動作描述 | 運鏡 | 旁白 / 台詞 | 字幕 | 參考圖 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 01_opening | 0-8s | station | A young female clinician on the left holds a clipboard and presents calmly while a senior male clinician on the right listens and nods attentively in a bright clinical team room | Steady medium two-shot keeping the 180-degree camera axis locked | 大家好，今天我們用一分鐘的時間，帶大家掌握結構化交班的四個核心步驟。 | 今天帶大家掌握結構化交班的四個核心步驟 | scene_anchor.png |
| 02_situation | 8-18s | station | Continuing in the same room and screen positions, the female clinician on the left points to the key section on the clipboard while the senior clinician on the right nods in agreement | Very slow push-in to a medium close two-shot | 第一步是說明目前狀況與關鍵背景，讓接手團隊在最短時間內抓到重點。 | 第一步：說明目前狀況與關鍵背景，讓團隊快速抓到重點 | scene_anchor.png |
| 03_assessment | 18-28s | station | The female clinician on the left gestures two clear points with a composed expression while the senior clinician on the right takes notes and smiles | Steady medium two-shot with consistent lighting and background | 第二步是提出精準評估與具體建議，明確告訴團隊接下來需要配合的處置方向。 | 第二步：提出精準評估與具體建議，明確說明處置方向 | scene_anchor.png |
| 04_closing | 28-36s | station | Both clinicians finish the handoff; the female clinician closes the clipboard and both smile naturally toward the camera | Steady medium two-shot holding the final pose cleanly | 掌握結構、穩定表達，就能讓每一次團隊溝通既安全又高效率！ | 掌握結構、穩定表達，讓團隊溝通既安全又高效率！ | scene_anchor.png |
'''

from mvkit.script_loader import (
    h3_frames,
    load_storyboard,
    load_storyboard_from_markdown,
    preview_shot,
)

if STORYBOARD_SOURCE_TYPE == "upload_file":
    from google.colab import files
    uploaded = files.upload()
    up_name = next(iter(uploaded.keys()))
    up_path = WORKSPACE_DIR / up_name
    up_path.write_bytes(uploaded[up_name])
    STORYBOARD_FILE_PATH = str(up_path)
    STORYBOARD_SOURCE_TYPE = "file_path"

if STORYBOARD_SOURCE_TYPE == "markdown_inline":
    project = load_storyboard_from_markdown(
        MARKDOWN_STORYBOARD,
        title=PROJECT_TITLE,
        audio_mode=AUDIO_MODE,
        continuity_mode=CONTINUITY_MODE,
        default_reference_image=str(default_anchor),
    )
else:
    project = load_storyboard(
        STORYBOARD_FILE_PATH,
        audio_mode=AUDIO_MODE,
        continuity_mode=CONTINUITY_MODE,
        default_reference_image=str(default_anchor),
    )
    project.title = PROJECT_TITLE

project.set_aspect(ASPECT_RATIO)
project.inference_mode = INFERENCE_MODE
project.ref_image_size = REF_IMAGE_SIZE
project.tail_guard_sec = float(TAIL_GUARD_SEC)
project.base_seed = int(BASE_SEED)
if MASTER_AUDIO_PATH.strip():
    project.master_audio_path = MASTER_AUDIO_PATH.strip()

project.validate()

print(f"✅ 分鏡腳本驗證通過：{project.title}（共 {len(project.shots)} 鏡，畫布 {project.width}×{project.height} @ 24fps）")
print(f"   - 單一聲軌模式 (audio_mode)     : {project.audio_mode}")
print(f"   - 畫面連貫模式 (continuity_mode): {project.continuity_mode} (tail_guard={project.tail_guard_sec}s)")
print(f"   - 推論模式 (inference_mode)     : {project.inference_mode} (ref_image_size={project.ref_image_size})")
for s in project.shots:
    gen_s = project.generation_seconds(s)
    print(f"   • [{s.index:02d}] {s.shot_id}: 成品保留 {s.duration_sec:.2f}s | H3 生成 {gen_s:.2f}s ({h3_frames(gen_s)} 幀)")

print("\n--- 第一鏡實際送入 MiniMax H3 的完整 Prompt 預覽 ---")
print(preview_shot(project, project.shots[0], base_dirs=[REFS_DIR]))

In [ ]:
#@title Step 4. 準備 MiniMax H3 權重、啟動 ComfyUI 並依分鏡批次生成影片（支援斷點續傳與單鏡重抽）
SAVE_DOWNLOADED_MODELS_TO_DRIVE = False #@param {type:"boolean"}
LOW_VRAM_MODE = "auto" #@param ["auto", "on", "off"]
RESUME_FROM_CHECKPOINT = True #@param {type:"boolean"}
ONLY_SHOT_IDS = "" #@param {type:"string"}

from mvkit.h3_colab_runner import H3ColabRunner

model_roots = [Path(DRIVE_ROOT) / "models"] if MOUNT_GOOGLE_DRIVE else []
runner = H3ColabRunner(
    comfy_dir=COMFY,
    base_url="http://127.0.0.1:8188",
    model_roots=model_roots,
)

models = runner.detect_models(project.inference_mode)
print(f"選用模型組合：mode={models.mode}, variant={models.variant}, diffusion={models.diffusion}, steps={models.steps}")

download_target = (Path(DRIVE_ROOT) / "models") if (MOUNT_GOOGLE_DRIVE and SAVE_DOWNLOADED_MODELS_TO_DRIVE) else None
runner.ensure_models(download_to=download_target)
runner.start_server(low_vram=LOW_VRAM_MODE)

only_list = [x.strip() for x in ONLY_SHOT_IDS.split(",") if x.strip()] or None
raw_clips = runner.run_storyboard(
    project,
    OUTPUT_DIR / project.title,
    base_dirs=[REFS_DIR, Path(STORYBOARD_FILE_PATH).parent],
    resume=RESUME_FROM_CHECKPOINT,
    only=only_list,
)
print(f"🎉 全部 {len(raw_clips)} 個分鏡片段已就緒！")

In [ ]:
#@title Step 5. 單一聲軌正規化、自動繁體中文字幕對齊與燒錄、品質檢視表與下載
USE_ASR_ALIGNMENT = False #@param {type:"boolean"}
BURN_SUBTITLES = True #@param {type:"boolean"}
AUTO_DOWNLOAD_MP4 = False #@param {type:"boolean"}

import shutil
from IPython.display import Image as IPyImage, Video, display
from mvkit.colab_pipeline import assemble_movie

# 先請 ComfyUI 釋放 VRAM，確保後製與字幕處理穩定
runner.free_vram()

manifest = assemble_movie(
    project,
    raw_clips,
    OUTPUT_DIR / project.title,
    base_dir=AUDIO_DIR,
    use_asr_alignment=USE_ASR_ALIGNMENT,
    asr_device="cpu",
    burn_subtitles=BURN_SUBTITLES,
)

print("============================================================")
print("✅ 成品輸出與「單一聲軌」驗證通過！")
print(f"   - 乾淨無字幕版 MP4 : {manifest['clean_mp4']}")
print(f"   - 燒錄繁中字幕 MP4 : {manifest['subtitled_mp4']}")
print(f"   - SRT 字幕檔       : {manifest['srt_path']}")
print(f"   - ASS 字幕檔       : {manifest['ass_path']}")
print(f"   - 連貫性檢視圖表   : {manifest['review_sheet']}")
a_info = manifest["clean_stream_verification"]["audio_stream"]
print(f"   - 音軌驗證結果     : 1 條視訊軌 + 1 條音訊軌 ({a_info['codec_name']}, {a_info['sample_rate']} Hz, {a_info['channels']} ch)")
print("============================================================")

if MOUNT_GOOGLE_DRIVE and Path(DRIVE_ROOT).exists():
    drive_out = Path(DRIVE_ROOT) / "outputs" / project.title
    drive_out.mkdir(parents=True, exist_ok=True)
    for key in ("clean_mp4", "subtitled_mp4", "srt_path", "ass_path", "review_sheet"):
        val = manifest.get(key)
        if val and Path(val).is_file():
            shutil.copy2(val, drive_out / Path(val).name)
    print(f"📂 已同步備份至 Google Drive：{drive_out}")

if manifest.get("review_sheet") and Path(manifest["review_sheet"]).is_file():
    display(IPyImage(filename=manifest["review_sheet"]))

final_video = manifest.get("subtitled_mp4") or manifest["clean_mp4"]
display(Video(str(final_video), embed=True))

if AUTO_DOWNLOAD_MP4:
    from google.colab import files
    files.download(str(final_video))
    if manifest.get("srt_path"):
        files.download(str(manifest["srt_path"]))